# Analyzing Potential Relationships between Permit Clusters and Economic Indicators

## Preliminaries

In [1]:
# Imports

# General

import numpy as np
import pandas as pd
import geopandas as gpd
import math
import json
import re
import string

# Plotting

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected' # For plotly graphs to render in this environment


# Scikit-Learn

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

In [20]:
# Set directory

PATH = "C:/Users/emshe/Desktop/BRAINSTATION/CAPSTONE/GIT_REPO/DATA"

In [21]:
# Load clustered builds

builds_df = pd.read_csv(f'{PATH}/CLUSTERED/builds_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(builds_df)

In [22]:
# Load clustered demos

demos_df = pd.read_csv(f'{PATH}/CLUSTERED/demos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(demos_df)

In [23]:
# Load clustered renos

renos_df = pd.read_csv(f'{PATH}/CLUSTERED/renos_clustered.csv',
                parse_dates=['issue_date']) # Force datetime format for date columns

# examine_df(renos_df)

In [24]:
# Load economic data

econ_df = pd.read_csv(f'{PATH}/ECONOMIC/PROCESSED/full_economic_data.csv')


# examine_df(econ_df)

### Helper functions

In [40]:
# Function to examine dataframes

def examine_df(df,
               name = 'dataframe',
               include_stats = True,
               include_sample = True):
    
    """
    Check basic info about a dataframe df
    """
    
    print(f"\n\nNumber of records in the {name} is: {len(df)}\n")
    print(f"The columns in the {name} are: {df.columns}\n")
    print(f"\n Other info about {name}:\n")
    display(df.info())
    if include_stats == True:
        print(f'\n Basic statistical info about {name}:\n')
        display(df.describe())
    if include_sample == True:
        print(f"\n\nSample of records in the {name}:")
        display(df.head(5))

In [15]:
# Function to generate correlation heatmap

def gen_corr_heatmap(df):

    '''
    Generate correlation heatmap for numeric columns of a dataframe
    '''
    
    num_df = df.select_dtypes(include = 'number') # Restrict to numeric columns
    corr_matrix = num_df.corr() # Compute correlation matrix
    
    fig = px.imshow(
        corr_matrix,
        text_auto=True, # Include text
        color_continuous_scale='RdBu', # Set color scale
        aspect='auto', # Set aspect ratio
        title='Correlation Heatmap of Numeric Columns',
        zmin=-1,   # force range
        zmax=1
    ) # Generate heatmap figure
    fig.update_layout(title={'x': 0.5})
    
    fig.update_layout(width=1100, height=1100)          # bigger figure
    fig.update_xaxes(tickangle=45, tickfont=dict(size=9))
    fig.update_yaxes(tickfont=dict(size=9))
        
    fig.show(renderer="notebook")

In [25]:
# Function to prepare clustered permits for statistical testing

def prep_permits(df: pd.DataFrame, dev_type: str) -> pd.DataFrame:

    """
    Prepare cluster permit data for statistical testing
    """
    
    out = df.copy()
    
    out['index'] = np.arange(len(out))

    # Keep original index too (uncomment if you want a back-reference)
    # out['src_index'] = df.index.to_numpy()

    # Date → year
    out['issue_date'] = pd.to_datetime(out['issue_date'], errors='coerce')
    out['year'] = out['issue_date'].dt.year

    # Dev type tag
    out['dev_type'] = dev_type  # e.g., {'build','reno','demo'}

    return out[['index','nbhd','year','cluster','dev_type']]

In [34]:
# Function to prepare economic data for statistical testing

def prep_econ(df: pd.DataFrame) -> pd.DataFrame:
    
    """
    Keep only neighborhood, zone, year, and change columns
    for correlation / regression analysis.
    """
    
    keep_cols = ['nbhd', 'zone', 'year'] + [c for c in df.columns if c.endswith('_change')]
    return df[keep_cols].copy()

### Preparing Data

In [30]:
# Get prepared clustered builds dataframe

prep_builds_df = prep_permits(builds_df, 'build')

# examine_df(prep_builds_df)

In [36]:
# Get prepared clustered renos dataframe

prep_renos_df = prep_permits(renos_df, 'reno')

# examine_df(prep_renos_df)

In [38]:
# Get prepared clustered demos dataframe

prep_demos_df = prep_permits(demos_df, 'demo')

# examine_df(prep_demos_df)

In [45]:
# Concatenate different permit ypes

prep_permits_df = pd.concat(
    [prep_builds_df, prep_renos_df, prep_demos_df],
    axis=0,
    ignore_index=True  # resets the index
)

# Add a globally unique permit UID
prep_permits_df['uid'] = (
    prep_permits_df['dev_type'].str[:1] +  # first letter of dev_type (b/r/d)
    prep_permits_df['index'].astype(str)
)

examine_df(prep_permits_df)



Number of records in the dataframe is: 23051

The columns in the dataframe are: Index(['index', 'nbhd', 'year', 'cluster', 'dev_type', 'uid'], dtype='object')


 Other info about dataframe:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23051 entries, 0 to 23050
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   index     23051 non-null  int32 
 1   nbhd      23051 non-null  object
 2   year      23051 non-null  int32 
 3   cluster   23051 non-null  int64 
 4   dev_type  23051 non-null  object
 5   uid       23051 non-null  object
dtypes: int32(2), int64(1), object(3)
memory usage: 900.6+ KB


None


 Basic statistical info about dataframe:



,index,year,cluster
count,23051.000000,23051.000000,23051.000000
mean,3994.876882,2020.495293,1.944731
std,2459.122690,2.470057,1.950800
min,0.000000,2017.000000,0.000000
25%,1920.500000,2018.000000,0.000000
50%,3841.000000,2020.000000,1.000000
75%,5786.500000,2023.000000,3.000000
max,9461.000000,2025.000000,7.000000




Sample of records in the dataframe:


,index,nbhd,year,cluster,dev_type,uid
0,0,Renfrew,2017,3,build,b0
1,1,Sunset,2017,3,build,b1
2,2,Renfrew,2017,2,build,b2
3,3,Fraser View/Killarny,2024,5,build,b3
4,4,Hastings/Sunrise/Grandview/Woodlands,2024,6,build,b4


In [42]:
# Get prepared economic dataframe

prep_econ_df = prep_econ(econ_df)

# examine_df(prep_econ_df)

## Computing Cluster Scores

In [ ]:
# Function to compute permit scores

import numpy as np
import pandas as pd

def compute_cluster_scores(
    permits: pd.DataFrame,
    by_year: bool = True,
    smooth: float = 1.0,
    return_wide: bool = False,
    which_scores: tuple = ("share", "tfidf")) -> pd.DataFrame:
    
    """
    Compute neighborhood-by-year cluster scores:
      - share_{dev}_cK: within-(nbhd, year, dev_type) share of cluster K
      - tfidf_{dev}_cK: share * log(1 / citywide_frequency_of_K) 
    """
    
    req = {'nbhd','year','dev_type','cluster'}
    missing = req - set(permits.columns)
    if missing:
        raise ValueError(f"permits is missing required column(s): {missing}")

    df = permits.copy()

    # 1) Neighborhood-year-dev_type-cluster counts
    g = (
        df.groupby(['nbhd','year','dev_type','cluster'])
          .size().rename('count').reset_index()
    )

    # Totals per neighborhood-year-dev_type
    tot = (
        g.groupby(['nbhd','year','dev_type'])['count']
         .sum().rename('total_devtype').reset_index()
    )
    out = g.merge(tot, on=['nbhd','year','dev_type'], how='left')

    # 2) Shares (composition)
    out['share'] = out['count'] / out['total_devtype'].replace({0: np.nan})

    # 3) Citywide rarity weights (IDF-like)
    #    Compute cluster counts at city level, either per year or across all years.
    if by_year:
        key_city = ['dev_type','year','cluster']
        key_total = ['dev_type','year']
    else:
        key_city = ['dev_type','cluster']
        key_total = ['dev_type']

    city_cluster = (
        df.groupby(key_city).size()
          .rename('city_cluster_count').reset_index()
    )
    city_total = (
        df.groupby(key_total).size()
          .rename('city_total_count').reset_index()
    )

    # Merge citywide counts back to out
    out = out.merge(city_cluster, on=key_city, how='left')
    out = out.merge(city_total, on=key_total, how='left')

    # IDF = log( (total + s) / (cluster + s) )  == log(1 / p_k) with smoothing
    out['idf'] = np.log((out['city_total_count'] + smooth) /
                        (out['city_cluster_count'] + smooth))

    # 4) TF–IDF style score
    out['tfidf'] = out['share'] * out['idf']

    # 5) Keep only requested scores
    keep_cols = ['nbhd','year','dev_type','cluster','count','total_devtype']
    if 'share' in which_scores: keep_cols.append('share')
    if 'tfidf' in which_scores: keep_cols += ['idf','tfidf']
    out = out[keep_cols]

    if not return_wide:
        return out

    # 6) Wide pivot for easy merging with econ panel
    long_for_pivot = out.melt(
        id_vars=['nbhd','year','dev_type','cluster'],
        value_vars=[c for c in ['share','tfidf'] if c in which_scores],
        var_name='score_type',
        value_name='score_value'
    )

    long_for_pivot['col'] = (
        long_for_pivot
        .apply(lambda r: f"{r['score_type']}_{r['dev_type']}_c{int(r['cluster'])}", axis=1)
    )

    wide = (
        long_for_pivot
        .pivot_table(index=['nbhd','year'], columns='col', values='score_value', aggfunc='first')
        .reset_index()
    )
    return wide
